[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/juliopez/Taller-Fundamentos-Data-Science-Python/blob/main/Machine_Learning_2026/03_Notebooks/NB06_Transfer_Learning_Fine_Tuning_Evaluacion_CNN.ipynb)
# NB06 — Experimentar con modelos preentrenados usando imágenes propias

## Propósito
En NB05 entrenaste una CNN para tus propias categorías. Ahora explorarás **modelos ya entrenados sobre ImageNet (más de un millón de imágenes, 1.000 categorías)**, sin entrenar ni ajustar sus pesos. Probaremos **MobileNetV2** y **EfficientNetB0** con imágenes de tu dataset y analizaremos qué reconocen y qué no.

**Importante:** las categorías ImageNet no son necesariamente las de tu dataset. Las predicciones de estos modelos no deben interpretarse como evaluación directa de tu clasificador NB05. No haremos Fine-Tuning en este notebook.

**Producto:** un notebook ejecutado con tabla de predicciones, observaciones y respuestas en Markdown.

## 1. Preparar el entorno

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from pathlib import Path
from PIL import Image
from IPython.display import display

print("TensorFlow:", tf.__version__)

## 2. Conectar Google Drive y localizar el dataset propio
Usa **la misma carpeta de imágenes clasificadas por subcarpetas** que en NB05. No necesitas volver a entrenar tu CNN.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATASET_DIR = Path("/content/drive/MyDrive/MachineLearning2026/dataset")  # ADAPTAR
assert DATASET_DIR.is_dir(), f"No existe la carpeta: {DATASET_DIR}"
CLASES_PROPIAS = sorted(p.name for p in DATASET_DIR.iterdir() if p.is_dir())
assert CLASES_PROPIAS, "No se encontraron subcarpetas de clases"
print("Categorías de mi dataset:", CLASES_PROPIAS)

## 3. Elegir imágenes reales de cada clase
Selecciona hasta **2 imágenes por clase** para una prueba breve. Si tienes muchas categorías, puedes limitar la muestra modificando `MAX_CLASES`. No existe un mínimo obligatorio de imágenes ni de clases para tu dataset.

In [ ]:
EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
MAX_CLASES = None  # por ejemplo, 5 para limitar el tiempo
IMAGENES_POR_CLASE = 2
registros = []
for nombre in CLASES_PROPIAS[:MAX_CLASES]:
    archivos = sorted(p for p in (DATASET_DIR / nombre).rglob("*") if p.is_file() and p.suffix.lower() in EXT)
    for ruta in archivos[:IMAGENES_POR_CLASE]:
        registros.append({"clase_propia": nombre, "ruta": str(ruta)})
assert registros, "No hay imágenes compatibles para analizar"
muestra = pd.DataFrame(registros)
display(muestra)

## 4. Cargar dos modelos preentrenados
Ambos fueron entrenados para reconocer categorías de ImageNet. **No se realiza entrenamiento**: solo inferencia. La primera descarga de pesos requiere conexión a Internet.

In [ ]:
mobilenet = keras.applications.MobileNetV2(weights="imagenet", include_top=True)
efficientnet = keras.applications.EfficientNetB0(weights="imagenet", include_top=True)
print("MobileNetV2:", mobilenet.count_params(), "parámetros")
print("EfficientNetB0:", efficientnet.count_params(), "parámetros")

## 5. Preprocesar correctamente para cada modelo
MobileNetV2 requiere `preprocess_input` (escala aproximada [-1, 1]). En **EfficientNetB0 de Keras**, el escalado está incorporado en la arquitectura y `preprocess_input` es una operación identidad; recibe píxeles en escala 0–255. No intercambies ambos preprocesamientos.

In [ ]:
TAMANO = (224, 224)
def cargar_rgb(ruta):
    with Image.open(ruta) as img:
        img = img.convert("RGB").resize(TAMANO)
        return np.asarray(img, dtype=np.float32)[None, ...]

def inferir_ambos(ruta):
    x = cargar_rgb(ruta)
    pm = mobilenet.predict(keras.applications.mobilenet_v2.preprocess_input(x.copy()), verbose=0)
    pe = efficientnet.predict(keras.applications.efficientnet.preprocess_input(x.copy()), verbose=0)
    dm = keras.applications.mobilenet_v2.decode_predictions(pm, top=3)[0]
    de = keras.applications.efficientnet.decode_predictions(pe, top=3)[0]
    return dm, de

## 6. Comparar las predicciones en imágenes propias
La confianza mostrada corresponde a probabilidades **entre las 1.000 clases de ImageNet**, no a la probabilidad de acertar una categoría propia.

In [ ]:
resultados = []
for fila in muestra.itertuples(index=False):
    pred_m, pred_e = inferir_ambos(fila.ruta)
    for nombre_modelo, preds in [("MobileNetV2", pred_m), ("EfficientNetB0", pred_e)]:
        resultados.append({
            "clase_propia": fila.clase_propia,
            "imagen": Path(fila.ruta).name,
            "modelo": nombre_modelo,
            "top1_imagenet": preds[0][1],
            "confianza_top1": float(preds[0][2]),
            "top3_imagenet": " | ".join(f"{nombre} ({prob:.1%})" for _, nombre, prob in preds)
        })
tabla = pd.DataFrame(resultados)
display(tabla)

## 7. Observar imágenes y resultados
Inspecciona algunas imágenes para interpretar las etiquetas propuestas por los modelos.

In [ ]:
for fila in muestra.head(4).itertuples(index=False):
    print("Clase propia:", fila.clase_propia, "| Imagen:", Path(fila.ruta).name)
    display(Image.open(fila.ruta).convert("RGB").resize((180, 180)))
    display(tabla[tabla["imagen"] == Path(fila.ruta).name][["modelo", "top1_imagenet", "confianza_top1"]])

## 8. Actividad de análisis — responder en celdas Markdown

**A.** ¿Qué diferencia existe entre tu CNN del NB05, entrenada con tus propias categorías, y estos modelos preentrenados en ImageNet?

**B.** Identifica al menos **dos predicciones concretas** de la tabla: ¿qué modelo las produjo?, ¿qué imagen recibió?, ¿qué etiqueta propuso? ¿Son semánticamente útiles para tu problema?

**C.** ¿En qué casos las categorías de ImageNet no coinciden con las categorías de tu dataset? Explica por qué un nivel de confianza alto no garantiza que la predicción sea útil o correcta para tu problema.

**D.** Compara MobileNetV2 y EfficientNetB0 en tu muestra. ¿Qué diferencias observas? Evita concluir que uno es universalmente superior basándote en pocas imágenes.

**E.** ¿Qué permitiría hacer el *Transfer Learning* para adaptar un modelo preentrenado a tus categorías? Explica la idea, **sin implementarlo**.

## 9. Cierre y continuidad

En este NB06 hemos utilizado modelos preentrenados **sin reentrenarlos**. Las predicciones ImageNet sirven para comprender sus capacidades y limitaciones, no para reemplazar automáticamente la CNN específica del NB05.

**Próxima clase (NB07):** trabajaremos con un **modelo común ya entrenado** de piedra, papel o tijera, para estudiar su preparación e interoperabilidad antes del despliegue en NB08. No es necesario que conviertas tus modelos propios a ONNX en esta clase.